In [ ]:
import sys
import os
curPath = os.path.abspath(os.path.dirname('detection2'))
print(curPath)
rootPath = os.path.split(curPath)[0]
sys.path.append(rootPath)
import torch
import torch as t
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchnet import meter
import xarray as xr
import rioxarray as rxr
from torch.nn import functional as F
import math
from models.Conv1d_transformer import  transformer_conv1d,Transformer_Muti_kernel_Conv1d, transformer_mlp,LSTM_conv1d
from models.Conv1d_transformer import Inception_time
from models.STSCDT import Transformer_MKConv1d2
from models.STSCDT import *
from models.Conv1d_transformer import *
from models.LSTM import BiLSTMModel, BiGRUModel, BiGRUModel_revise, BiLSTMModel_revise
from models.TCN import TCN
from deeplearning.net import *
from load_data import *
from deeplearning.params import *
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
#--------------------------for STS_CDT_Revise_all_3.py ---------------------------

param_no = 1
region = 'RSG'
province =  'SD'
folder_path = f'G:/Sentinel-SAR/{province}/{region}/blocks_revise/'
# folder_path = f'D:/Get_result/{province}/{region}/blocks_revise/'

patch_num = count_tif_files(folder_path)

for i in range(1,patch_num+1):
    
    no = i
    print(i,'/',patch_num, '%')

    # PATH = f"D:/Get_result/{province}/{region}/blocks_revise/{region}_R{no}.tif"
    # output_path = f'D:/Get_result/{province}/{region}/memory/{region}_R{no}_E{param_no}.tif'
    PATH = f'G:/Sentinel-SAR/{province}/{region}/blocks_revise/{region}_R{no}.tif'
    output_path = f'G:/Sentinel-SAR/{province}/{region}/memory/{region}_R{no}_E{param_no}.tif'
       
#----------------------------------model--------------------------------------------------------
    new_net = STS_CTD3(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6, ff_h= 256,
                  conv_channels=[128, 64, 32, 16, 1], seq_len = 27, bandDropout=0).to(device) 
    param_path = 'E:/min/detection2/model_params/STS_CTD_Revise_all_3_e200.py'
    #param_path = 'E:\min\detection2/revise_params\STSCDT_revise_all_e200_2.py'
#-----------------------------------------------------------------------------------------------
    # new_net = STS_CTD3_MKD(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6, ff_h= 256,
    #               conv_channels=[128, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'E:/min/detection2/model_params/STS_CDT_Revise_all_3_e200_MKD.py'
# #-----------------------------------------------------------------------------------------------
    # new_net = STS_CTD3_SKBD(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6, ff_h= 256,
    #               conv_channels=[128, 64, 32, 16, 1], seq_len = 27, bandDropout = 0).to(device) 
    # param_path = 'E:/min/detection2/model_params/STS_CDT_Revise_all_3_e200_SKBD.py'
# #-----------------------------------------------------------------------------------------------
    # new_net = Transformer_revise(d_model = 128, d_k= 16, heads=8, dropout=0.5, norm_shape=[27, 128], ff_h=256, num_encoder=6, mlp = 256, mlp2=128).to(device) 
    # param_path = 'E:/min/detection2/model_params/STS_CDT_Revise_all_transformer.py'
    
#-----------------------------------------------------------------------------------------------
    # new_net = STS_CTD3_MKNBP(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6, ff_h= 256,
    #                conv_channels=[128, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'E:/min/detection2/model_params/STS_CDT_Revise_all_3_result_e200_MKNBP.py'

#-----------------------------------------Baseline model ------------------------------------------
    # new_net = BiLSTMModel_revise(input_size=7, hidden_size=256, num_layers=4, output_size=1).to(device)
    # param_path = 'E:\min\detection2/revise_params/BiLSTM_revise_e200.py'

#-------------------------------------------------------------------------------------------------------
    # new_net = BiGRUModel_revise(input_size=7, hidden_size=128, num_layers=4, output_size=1).to(device)
    # param_path = 'E:\min\detection2/revise_params/BiGRU_revise_e200.py'

#-------------------------------------------------------------------------------------------------------
    # new_net = TCN(input_size=7, output_size=1, num_channels=[72, 48, 36, 30, 24, 18, 12, 6, 6, 3]).to(device) 
    # param_path = 'E:\min\detection2/revise_params/TCN_revise_e200.py'

# #-------------------------------------------------------------------------------------------------------
    # new_net = Inception_time(in_channels=7, out_channel=32, kernel_sizes=[1, 3, 5], bottleneck_channels = 32).to(device)
    # param_path = 'E:\min\detection2/revise_params/Inceptiom_revise_e200.py'
    
#-------------------------------------------------------------------------------------------------------

    raster = rxr.open_rasterio(PATH).values
    print(raster.shape)
    raster_row, raster_col = (raster.shape)[1], (raster.shape)[2]
        #print( raster_row, raster_col)
        
    row = raster_row
    interval = 10
    cols = [[i, i+interval] for i in range(0, raster_col, interval)]
    if cols[-1][1] != raster_col:
        cols[-1][1] = raster_col

    rows_and_cols = [[0, row]] * len(cols)
    rows_and_cols = [[r, c] for r, c in zip(rows_and_cols, cols)]

    row1, col1 = rows_and_cols[0]
    row2, col2 = rows_and_cols[1]
    is_remove_band = False
    target_band = 'vvhh_exceppt'
    a = []
    for i in range(len(cols)):
        row, col = rows_and_cols[i]
        r = get_result_revise_2(PATH, row, col, new_net, param_path, device)
        a.append(r)
        result = np.hstack(a)
        #result = result*365
        # #print('result.shape:',result.shape)
        # save_img(output_path, PATH, result)
    save_img(output_path, PATH, result)

#---------------------------------------------------------------------------

In [ ]:
#-----------------------------Validation--------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from sklearn.metrics import r2_score



province = 'GX'
PATH = f'E:/Sentinel-SAR/Train_data_all/{province}_test_data7_revise.tif'
# PATH = 'E:/Sentinel-SAR/Train_data_all/HB_SD_JS_SH_ZJ_FJ_GX_test_data7_revise.tif'

# result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_{province}_3_result_e200_NBD.tif'
result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_{province}_3_result_e200.tif'
#-----------------------------------------------------------------------------------------------
# result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200_MKD.tif'

# result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200_SKBD.tif'
# result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200_MLP.tif'
#result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200_NBD.tif'
# result_path = 'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200_MKNBP.tif'

# result_path = f'E:/min/detection2/model_params/STS_CDT_Test_Revise_all_3_result_e200.tif'
# result_path = 'E:\min\detection2/revise_params/BiLSTM_test_revise.tif'
# result_path = 'E:\min\detection2/revise_params/BiGRU_test_revise.tif'
# result_path = 'E:\min\detection2/revise_params/TCN_test_revise.tif'
# result_path = 'E:\min\detection2/revise_params/Inception_test_revise.tif'
#result_path = 'E:/min/detection2/model_params/Transformer_revise_result.tif'

#--------------------------------Ablation experiments for Feature bands-------------------------------------
# result_path = 'E:/min/detection2/model_params/STSCTD_quitCR_revise_resulta.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitVV_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitVH_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitVVVH_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_onlyVVVH_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitRVI_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitVDDPI_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STSCTD_quitDpRVIc_revise_result.tif'
# result_path = 'E:/min/detection2/model_params/STS_CTD_Test_Revise_all_3_result_e200.tif'
#result_path = 'E:/min/detection2/model_params/STS_CTD_Test_Revise_all_3_result_e200_MKD2.tif'

result = (rxr.open_rasterio(result_path).values).squeeze(0)
test = rxr.open_rasterio(PATH).values
label = test[-1,:,:]
last_doy = test[-2,:,:]
print(label.shape, result.shape)

t_label = torch.tensor(label).unsqueeze(0)
t_result = torch.tensor(result.round()).unsqueeze(0)
combine = torch.cat([t_label, t_result], dim = 0)
print(combine.shape)
combine.shape
mask = torch.logical_or(combine[0, :, :] < 367, combine[1, :, :] < 367)
mask = combine[0, :, :] < 367
q = combine[:, mask]  
print(q.shape)
label, result = q[0, :], q[1,:]


In [ ]:
x = label.reshape(-1,1).squeeze(1)
result[result>=367] = 367
y = result.reshape(-1,1).squeeze(1)
# random_indices = random.sample(range(20000), 5000)
# y = np.array(y[random_indices])
# x = np.array(x[random_indices])
y = np.array(y)
x = np.array(x)
R, _ = pearsonr(x, y)
R2 = r2_score(y, x)
MAD = np.mean(np.abs(y-x))
RMSE = np.sqrt(np.mean((y-x)**2))
MSB = np.mean((y-x))
# MSE = np.mean((y-x)**2)


xy = np.vstack([x,y])  
z = gaussian_kde(xy)(xy)  

# Sort the points by density, so that the densest points are plotted last
idx = z.argsort()
x, y, z = x[idx], y[idx], z[idx]

fig, ax = plt.subplots()
#textstr = f"R = {R:.2f}\nMAD = {MAD:.2f}\nRMSE = {RMSE:.2f}\nMSB = {MSB:.2f}"
textstr = f"R$^2$ = {R2:.2f}\nMAE = {MAD:.2f}\nRMSE = {RMSE:.2f}\nMBE = {MSB:.2f}"
ax.text(0.05, 0.95, textstr, transform = ax.transAxes, fontsize = 16, verticalalignment = 'top')

ax.set_xlabel('Label DOY', fontsize = 14)
ax.set_ylabel('Estimated DOY', fontsize = 14)
#plt.scatter(x, y,c=z, s=2,cmap='YlGnBu') # c表示标记的颜色Spectral_r

# plt.scatter(x, y,c=z, s=5, cmap='PuBu')
# plt.colorbar()
ax.plot([150, 400], [150, 400], color='gray', linestyle='--', linewidth=0.8, zorder=1)
sc = plt.scatter(x, y, c=z, s=5, cmap='PuBu', vmin=0, vmax=0.001, zorder=2)
cbar = plt.colorbar(sc)


# cbar.set_label('Density', fontsize=12)


# plt.title('SD')
ax.set_xlim(150, 400)
ax.set_ylim(150, 400)
ax.tick_params(axis='both', labelsize=12)
plt.show()